In [1]:
# check GPU
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Turn on GPU: Runtime > Change runtime type > T4 GPU")

Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
# connect Kaggle using API token
!pip install -q -U kaggle

from getpass import getpass
from pathlib import Path
import os

token = getpass("Paste Kaggle API token here: ")

Path("/root/.kaggle").mkdir(parents=True, exist_ok=True)
Path("/root/.kaggle/access_token").write_text(token.strip())
os.chmod("/root/.kaggle/access_token", 0o600)

os.environ["KAGGLE_API_TOKEN"] = token.strip()

!kaggle datasets list -s "NSynth WAV"

Paste Kaggle API token here: ··········
ref                                           title                                 size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------  -----------------------------  -----------  --------------------------  -------------  ---------  ---------------  
accelotron/nsynth-wav                         NSynth WAV+JSON                25069610428  2022-06-15 01:32:44.780000            618         12  0.875            
daemonkerrigan/nsynth-numerical-features-csv  NSynth Numerical Features CSV     78537224  2025-05-07 16:25:13.660000             19          0  0.47058824       
hanshiromase/nsynth-music                     Nsynth Music                   25079997212  2025-04-03 13:54:15.003000             10          0  0.3125           
alexdaniels88/nsynth-train                    nsynth-train                   25079997212  2025-05-19 10:19:55.513000              2          0  0.25  

In [8]:
# clean and download ZIP only, do NOT unzip full dataset
!rm -rf /content/kaggle_data /content/nsynth_subset
!mkdir -p /content/kaggle_data /content/nsynth_subset/audio

!kaggle datasets download -d accelotron/nsynth-wav -p /content/kaggle_data

Dataset URL: https://www.kaggle.com/datasets/accelotron/nsynth-wav
License(s): Attribution 4.0 International (CC BY 4.0)
100% 23.3G/23.3G [05:10<00:00, 80.6MB/s]



In [9]:
# extract only a small subset from the Kaggle ZIP
from pathlib import Path
import zipfile, shutil

ZIP_PATH = Path("/content/kaggle_data/nsynth-wav.zip")
OUT_AUDIO = Path("/content/nsynth_subset/audio")
OUT_AUDIO.mkdir(parents=True, exist_ok=True)

MAX_FILES = 2000

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    wavs = [n for n in z.namelist() if n.endswith(".wav") and "/audio/" in n]
    print("Total wavs in zip:", len(wavs))

    for i, name in enumerate(wavs[:MAX_FILES]):
        target = OUT_AUDIO / Path(name).name
        with z.open(name) as src, open(target, "wb") as dst:
            shutil.copyfileobj(src, dst)

        if (i + 1) % 500 == 0:
            print("Extracted:", i + 1)

print("Done. Extracted files:", len(list(OUT_AUDIO.glob("*.wav"))))

Total wavs in zip: 305979
Extracted: 500
Extracted: 1000
Extracted: 1500
Extracted: 2000
Done. Extracted files: 2000


In [10]:
#  settings
from pathlib import Path

INPUT_ROOT = Path("/content/nsynth_subset/audio")
WORK = Path("/content")
AUDIOCRAFT = WORK / "audiocraft"
EGS = AUDIOCRAFT / "egs" / "saraga_icm"

PROMPTS = [
    "Indian classical inspired plucked string note with tanpura-like drone texture",
    "Carnatic inspired violin phrase with sustained acoustic tone",
    "Hindustani inspired bansuri-like melodic phrase with soft drone",
    "Veena inspired plucked string improvisation with raga-like ornamentation",
    "Sitar-like acoustic plucked string phrase with Indian classical mood",
    "Meditative Indian classical instrumental texture with sustained notes",
]

print("INPUT_ROOT:", INPUT_ROOT)
print("Exists:", INPUT_ROOT.exists())
print("WAV files:", len(list(INPUT_ROOT.glob("*.wav"))))

INPUT_ROOT: /content/nsynth_subset/audio
Exists: True
WAV files: 2000


In [11]:
#  install AudioCraft
!apt-get update -qq
!apt-get install -y -qq ffmpeg

!git clone https://github.com/facebookresearch/audiocraft.git /content/audiocraft
%cd /content/audiocraft

!pip install -q setuptools wheel
!pip install -q av julius flashy hydra-core omegaconf einops encodec num2words spacy torchdiffeq
!pip install -q xformers
!pip install -q -e .

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
fatal: destination path '/content/audiocraft' already exists and is not an empty directory.
/content/audiocraft
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [13]:
# BROADER FIX CELL
!pip install -q \
  torchmetrics transformers sentencepiece \
  av julius flashy hydra-core omegaconf einops encodec num2words spacy torchdiffeq \
  librosa soundfile pandas scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 23.4 MB/s eta 0:00:00


In [14]:
#  test AudioCraft import
%cd /content/audiocraft

import torch
from audiocraft.models import MusicGen

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("AudioCraft import OK")

/content/audiocraft
Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
AudioCraft import OK


In [12]:
# prepare AudioCraft dataset manifests
import gzip, json, os, random, shutil, wave
from pathlib import Path

MAX_FILES = 2000
MIN_DURATION = 1.0

def wav_info(path):
    with wave.open(str(path), "rb") as w:
        sr = w.getframerate()
        channels = w.getnchannels()
        frames = w.getnframes()
    return frames / sr, sr, channels

def stage_audio(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.symlink(src, dst)
    except Exception:
        shutil.copy2(src, dst)

def write_manifest(split, rows):
    folder = EGS / split
    folder.mkdir(parents=True, exist_ok=True)
    with gzip.open(folder / "data.jsonl.gz", "wt", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

all_files = sorted(INPUT_ROOT.glob("*.wav"))
random.seed(7)
random.shuffle(all_files)
files = all_files[:MAX_FILES]

rows = []
audio_out = EGS / "audio"
audio_out.mkdir(parents=True, exist_ok=True)

for src in files:
    try:
        duration, sr, channels = wav_info(src)
    except Exception:
        continue

    if duration < MIN_DURATION:
        continue

    dst = audio_out / src.name
    stage_audio(src, dst)

    parts = src.stem.split("_")
    family = parts[0] if len(parts) > 0 else "instrument"
    source = parts[1] if len(parts) > 1 else "unknown"

    meta = {
        "title": src.stem,
        "artist": "nsynth",
        "key": None,
        "bpm": None,
        "genre": "instrumental",
        "moods": ["experimental", "minimal"],
        "keywords": [family, source, "single note", "instrument timbre"],
        "description": f"single sustained {source} {family} instrument note, clean isolated timbre",
        "name": "nsynth_subset",
        "instrument": family
    }
    dst.with_suffix(".json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    rows.append({
        "path": str(dst),
        "duration": round(duration, 3),
        "sample_rate": sr,
        "amplitude": None,
        "weight": None,
        "info_path": None
    })

random.seed(7)
random.shuffle(rows)

n = len(rows)
valid_n = max(1, int(n * 0.1))
eval_n = max(1, int(n * 0.1))

train = rows[: n - valid_n - eval_n]
valid = rows[n - valid_n - eval_n : n - eval_n]
evaluate = rows[n - eval_n :]

write_manifest("train", train)
write_manifest("valid", valid)
write_manifest("evaluate", evaluate)
write_manifest("generate", evaluate[:50])

print("prepared:", len(rows))
print("train:", len(train))
print("valid:", len(valid))
print("evaluate:", len(evaluate))
print("EGS:", EGS)

prepared: 2000
train: 1600
valid: 200
evaluate: 200
EGS: /content/audiocraft/egs/saraga_icm


In [13]:
# create AudioCraft dataset config
from pathlib import Path

config_dir = AUDIOCRAFT / "config" / "dset" / "audio"
config_dir.mkdir(parents=True, exist_ok=True)

(config_dir / "saraga_icm.yaml").write_text("""# @package __global__

datasource:
  max_sample_rate: 48000
  max_channels: 2
  train: egs/saraga_icm/train
  valid: egs/saraga_icm/valid
  evaluate: egs/saraga_icm/evaluate
  generate: egs/saraga_icm/generate
""")

print((config_dir / "saraga_icm.yaml").read_text())

# @package __global__

datasource:
  max_sample_rate: 48000
  max_channels: 2
  train: egs/saraga_icm/train
  valid: egs/saraga_icm/valid
  evaluate: egs/saraga_icm/evaluate
  generate: egs/saraga_icm/generate



In [14]:
#  clear broken AudioCraft import from memory
import sys

for name in list(sys.modules.keys()):
    if name == "audiocraft" or name.startswith("audiocraft."):
        del sys.modules[name]

import audiocraft
audiocraft.__version__ = getattr(audiocraft, "__version__", "1.4.0a2")

from audiocraft.models import MusicGen

print("AudioCraft fixed")
print("version:", audiocraft.__version__)

AudioCraft fixed
version: 1.4.0a2


In [23]:
#  baseline generation before fine-tuning

import json
import random
import torch
from pathlib import Path
from audiocraft.models import MusicGen
from audiocraft.data.audio import audio_write

try:
    PROMPTS
except NameError:
    PROMPTS = [
        "single sustained acoustic bass instrument note clean isolated timbre",
        "single sustained electronic bass instrument note clean isolated timbre",
        "single sustained guitar instrument note clean isolated timbre",
        "single sustained flute instrument note clean isolated timbre",
    ]

def safe_name(text):
    return "".join(c if c.isalnum() else "_" for c in text.lower())[:70]

def make_samples(
    model_path,
    out_dir,
    prompts,
    duration=8,
    seed=7,
    top_k=250,
    temperature=1.0,
    cfg_coef=3.0
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = MusicGen.get_pretrained(model_path, device="cuda")

    print("Loaded model:", model_path)
    print("Original model.max_duration:", model.max_duration)


    if model.max_duration <= duration:
        model.max_duration = duration + 1

    safe_stride = min(2, model.max_duration / 2)

    print("Using model.max_duration:", model.max_duration)
    print("Using duration:", duration)
    print("Using extend_stride:", safe_stride)

    model.set_generation_params(
        duration=duration,
        use_sampling=True,
        top_k=top_k,
        temperature=temperature,
        cfg_coef=cfg_coef,
        extend_stride=safe_stride,
    )

    manifest = []

    for i, prompt in enumerate(prompts):
        print("Generating:", prompt)

        wav = model.generate([prompt])[0].cpu()

        base = out_dir / f"{i:02d}_{safe_name(prompt)}"

        audio_write(
            str(base),
            wav,
            model.sample_rate,
            strategy="loudness",
            loudness_compressor=True
        )

        manifest.append({
            "prompt": prompt,
            "path": str(base.with_suffix(".wav"))
        })

    (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))

    print("Wrote samples to", out_dir)
    return manifest

pre_manifest = make_samples(
    "facebook/musicgen-small",
    "/content/pretrained_samples",
    PROMPTS,
    duration=8
)

pre_manifest

Loading weights:   0%|          | 0/99 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loaded model: facebook/musicgen-small
Original model.max_duration: 30
Using model.max_duration: 30
Using duration: 8
Using extend_stride: 2
Generating: Indian classical inspired plucked string note with tanpura-like drone texture
Generating: Carnatic inspired violin phrase with sustained acoustic tone
Generating: Hindustani inspired bansuri-like melodic phrase with soft drone
Generating: Veena inspired plucked string improvisation with raga-like ornamentation
Generating: Sitar-like acoustic plucked string phrase with Indian classical mood
Generating: Meditative Indian classical instrumental texture with sustained notes
Wrote samples to /content/pretrained_samples


[{'prompt': 'Indian classical inspired plucked string note with tanpura-like drone texture',
  'path': '/content/pretrained_samples/00_indian_classical_inspired_plucked_string_note_with_tanpura_like_drone_.wav'},
 {'prompt': 'Carnatic inspired violin phrase with sustained acoustic tone',
  'path': '/content/pretrained_samples/01_carnatic_inspired_violin_phrase_with_sustained_acoustic_tone.wav'},
 {'prompt': 'Hindustani inspired bansuri-like melodic phrase with soft drone',
  'path': '/content/pretrained_samples/02_hindustani_inspired_bansuri_like_melodic_phrase_with_soft_drone.wav'},
 {'prompt': 'Veena inspired plucked string improvisation with raga-like ornamentation',
  'path': '/content/pretrained_samples/03_veena_inspired_plucked_string_improvisation_with_raga_like_ornamentati.wav'},
 {'prompt': 'Sitar-like acoustic plucked string phrase with Indian classical mood',
  'path': '/content/pretrained_samples/04_sitar_like_acoustic_plucked_string_phrase_with_indian_classical_mood.wav'},

In [21]:
# listen to baseline samples
from IPython.display import Audio, display
import json
from pathlib import Path

manifest = json.loads(Path("/content/pretrained_samples/manifest.json").read_text())

for item in manifest:
    print(item["prompt"])
    display(Audio(item["path"]))

Indian classical inspired plucked string note with tanpura-like drone texture


Carnatic inspired violin phrase with sustained acoustic tone


Hindustani inspired bansuri-like melodic phrase with soft drone


Veena inspired plucked string improvisation with raga-like ornamentation


Sitar-like acoustic plucked string phrase with Indian classical mood


Meditative Indian classical instrumental texture with sustained notes


In [25]:
#  set missing Colab environment variables for Dora
import os

os.environ["USER"] = "colab"
os.environ["AUDIOCRAFT_DORA_DIR"] = "/content/dora"

print("USER:", os.environ["USER"])
print("AUDIOCRAFT_DORA_DIR:", os.environ["AUDIOCRAFT_DORA_DIR"])

USER: colab
AUDIOCRAFT_DORA_DIR: /content/dora


In [30]:
# BROADER AUDIOCRAFT TRAINING DEPS
!pip install -q pesq pystoi torchmetrics transformers sentencepiece \
  av julius flashy hydra-core omegaconf einops encodec num2words spacy torchdiffeq \
  librosa soundfile pandas scipy

  Preparing metadata (setup.py) ... done


In [2]:
#  quick MusicGen fine-tune test
%cd /content/audiocraft

# Make sure Dora exists
!pip install -q dora-search

import os
os.environ["USER"] = "colab"
os.environ["AUDIOCRAFT_DORA_DIR"] = "/content/dora"

!USER=colab AUDIOCRAFT_DORA_DIR=/content/dora dora run solver=musicgen/musicgen_base_32khz \
  model/lm/model_scale=small \
  continue_from=//pretrained/facebook/musicgen-small \
  conditioner=text2music \
  dset=audio/saraga_icm \
  dataset.batch_size=1 \
  dataset.segment_duration=2 \
  dataset.num_workers=0 \
  optim.epochs=1 \
  optim.updates_per_epoch=5 \
  optim.lr=1e-5 \
  evaluate.every=0 \
  generate.every=0

/content/audiocraft
2026-05-30 05:31:28.242211: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Dora directory: /content/dora
/usr/local/lib/python3.12/dist-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
[05-30 05:31:41][dora.distrib][INFO] - world_size is 1, skipping init.
[05-30 05:31:41][flashy.solver][INFO] - Instantiating solver MusicGenSolver for XP 50442320
[05-30 05:31:41][flashy.solver][INFO] - All XP logs are stored in /content/dora/xps/50442320
[05-30 05:31:41][audiocraft.solvers.builders][INFO] - Loading audio data split train:

In [3]:
#  find Dora experiment signature
from pathlib import Path

dora_dir = Path("/content/dora")
for p in dora_dir.rglob("checkpoint.th"):
    print("checkpoint:", p)
    print("possible signature folder:", p.parent.name)

checkpoint: /content/dora/xps/50442320/checkpoint.th
possible signature folder: 50442320


In [5]:
#  export trained model - FIXED for PyTorch 2.6
%cd /content/audiocraft

SIG = "50442320"

from pathlib import Path
from audiocraft import train
from audiocraft.utils import export
import torch

export_dir = Path("/content/musicgen_nsynth_small")
export_dir.mkdir(parents=True, exist_ok=True)

xp = train.main.get_xp_from_sig(SIG)

ckpt = xp.folder / "checkpoint.th"

print("XP folder:", xp.folder)
print("Checkpoint:", ckpt)
print("Checkpoint exists:", ckpt.exists())

# Patch torch.load so AudioCraft can load old-style full checkpoints
_original_torch_load = torch.load

def patched_torch_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _original_torch_load(*args, **kwargs)

torch.load = patched_torch_load

export.export_lm(
    ckpt,
    export_dir / "state_dict.bin"
)

export.export_pretrained_compression_model(
    "facebook/encodec_32khz",
    export_dir / "compression_state_dict.bin",
)

# Restore original torch.load
torch.load = _original_torch_load

print("exported to", export_dir)

/content/audiocraft
XP folder: /content/dora/xps/50442320
Checkpoint: /content/dora/xps/50442320/checkpoint.th
Checkpoint exists: True
exported to /content/musicgen_nsynth_small


In [6]:
!ls -lh /content/musicgen_nsynth_small

total 802M
-rw-r--r-- 1 root root 1.5K May 30 06:05 compression_state_dict.bin
-rw-r--r-- 1 root root 802M May 30 06:05 state_dict.bin


In [27]:
# generate after fine-tuning - FINAL PATCHED VERSION

%cd /content/audiocraft

import json
import random
import torch
from pathlib import Path
from audiocraft.models import MusicGen
from audiocraft.models.musicgen import MusicGen as MusicGenClass
from audiocraft.data.audio import audio_write

# Patch AudioCraft set_generation_params before get_pretrained()
_original_set_generation_params = MusicGenClass.set_generation_params

def patched_set_generation_params(
    self,
    use_sampling=True,
    top_k=250,
    top_p=0.0,
    temperature=1.0,
    duration=4,
    cfg_coef=3.0,
    cfg_coef_beta=None,
    two_step_cfg=False,
    extend_stride=18,
):
    # Fix bad max_duration in exported fine-tuned model
    if getattr(self, "max_duration", 0) <= extend_stride:
        self.max_duration = max(duration + 1, extend_stride + 1)

    return _original_set_generation_params(
        self,
        use_sampling=use_sampling,
        top_k=top_k,
        top_p=top_p,
        temperature=temperature,
        duration=duration,
        cfg_coef=cfg_coef,
        cfg_coef_beta=cfg_coef_beta,
        two_step_cfg=two_step_cfg,
        extend_stride=extend_stride,
    )

MusicGenClass.set_generation_params = patched_set_generation_params


def safe_name(text):
    return "".join(c if c.isalnum() else "_" for c in text.lower())[:70]


out_dir = Path("/content/finetuned_samples")
out_dir.mkdir(parents=True, exist_ok=True)

random.seed(7)
torch.manual_seed(7)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(7)

# Load fine-tuned exported model
model = MusicGen.get_pretrained("/content/musicgen_nsynth_small", device="cuda")

print("Loaded fine-tuned model")
print("Model max_duration after patch:", model.max_duration)

# Now set final generation params
model.set_generation_params(
    duration=4,
    use_sampling=True,
    top_k=250,
    temperature=1.0,
    cfg_coef=3.0,
    extend_stride=2,
)

post_manifest = []

for i, prompt in enumerate(PROMPTS):
    print("Generating:", prompt)

    wav = model.generate([prompt])[0].cpu()

    base = out_dir / f"{i:02d}_{safe_name(prompt)}"

    audio_write(
        str(base),
        wav,
        model.sample_rate,
        strategy="loudness",
        loudness_compressor=True
    )

    post_manifest.append({
        "prompt": prompt,
        "path": str(base.with_suffix(".wav"))
    })

(out_dir / "manifest.json").write_text(json.dumps(post_manifest, indent=2))

print("Wrote fine-tuned samples to", out_dir)

post_manifest

/content/audiocraft


Loading weights:   0%|          | 0/99 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/116 [00:00<?, ?it/s]

Loaded fine-tuned model
Model max_duration after patch: 19
Generating: Indian classical inspired plucked string note with tanpura-like drone texture
Generating: Carnatic inspired violin phrase with sustained acoustic tone
Generating: Hindustani inspired bansuri-like melodic phrase with soft drone
Generating: Veena inspired plucked string improvisation with raga-like ornamentation
Generating: Sitar-like acoustic plucked string phrase with Indian classical mood
Generating: Meditative Indian classical instrumental texture with sustained notes
Wrote fine-tuned samples to /content/finetuned_samples


[{'prompt': 'Indian classical inspired plucked string note with tanpura-like drone texture',
  'path': '/content/finetuned_samples/00_indian_classical_inspired_plucked_string_note_with_tanpura_like_drone_.wav'},
 {'prompt': 'Carnatic inspired violin phrase with sustained acoustic tone',
  'path': '/content/finetuned_samples/01_carnatic_inspired_violin_phrase_with_sustained_acoustic_tone.wav'},
 {'prompt': 'Hindustani inspired bansuri-like melodic phrase with soft drone',
  'path': '/content/finetuned_samples/02_hindustani_inspired_bansuri_like_melodic_phrase_with_soft_drone.wav'},
 {'prompt': 'Veena inspired plucked string improvisation with raga-like ornamentation',
  'path': '/content/finetuned_samples/03_veena_inspired_plucked_string_improvisation_with_raga_like_ornamentati.wav'},
 {'prompt': 'Sitar-like acoustic plucked string phrase with Indian classical mood',
  'path': '/content/finetuned_samples/04_sitar_like_acoustic_plucked_string_phrase_with_indian_classical_mood.wav'},
 {'p

In [28]:
#  listen to fine-tuned samples

from IPython.display import Audio, display
import json
from pathlib import Path

manifest_path = Path("/content/finetuned_samples/manifest.json")
manifest = json.loads(manifest_path.read_text())

for item in manifest:
    print(item["prompt"])
    display(Audio(item["path"]))

Indian classical inspired plucked string note with tanpura-like drone texture


Carnatic inspired violin phrase with sustained acoustic tone


Hindustani inspired bansuri-like melodic phrase with soft drone


Veena inspired plucked string improvisation with raga-like ornamentation


Sitar-like acoustic plucked string phrase with Indian classical mood


Meditative Indian classical instrumental texture with sustained notes


In [29]:
#  compare basic audio features

import json
import librosa
import pandas as pd
from pathlib import Path

pre_path = Path("/content/pretrained_samples/manifest.json")
post_path = Path("/content/finetuned_samples/manifest.json")

pre = json.loads(pre_path.read_text())
post = json.loads(post_path.read_text())

def features(path):
    y, sr = librosa.load(path, sr=None, mono=True)
    return {
        "duration": librosa.get_duration(y=y, sr=sr),
        "rms": float(librosa.feature.rms(y=y).mean()),
        "centroid": float(librosa.feature.spectral_centroid(y=y, sr=sr).mean()),
        "zcr": float(librosa.feature.zero_crossing_rate(y).mean()),
    }

rows = []

for a, b in zip(pre, post):
    fa = features(a["path"])
    fb = features(b["path"])

    rows.append({
        "prompt": a["prompt"],
        "pre_audio": a["path"],
        "post_audio": b["path"],
        "pre_duration": fa["duration"],
        "post_duration": fb["duration"],
        "pre_rms": fa["rms"],
        "post_rms": fb["rms"],
        "pre_centroid": fa["centroid"],
        "post_centroid": fb["centroid"],
        "pre_zcr": fa["zcr"],
        "post_zcr": fb["zcr"],
    })

df = pd.DataFrame(rows)
df.to_csv("/content/comparison.csv", index=False)

df

,prompt,pre_audio,post_audio,pre_duration,post_duration,pre_rms,post_rms,pre_centroid,post_centroid,pre_zcr,post_zcr
0,Indian classical inspired plucked string note ...,/content/pretrained_samples/00_indian_classica...,/content/finetuned_samples/00_indian_classical...,8.0,4.0,0.190540,0.185318,1761.998332,1565.374879,0.040185,0.042745
1,Carnatic inspired violin phrase with sustained...,/content/pretrained_samples/01_carnatic_inspir...,/content/finetuned_samples/01_carnatic_inspire...,8.0,4.0,0.192946,0.172320,1086.786910,1939.166379,0.032635,0.084613
2,Hindustani inspired bansuri-like melodic phras...,/content/pretrained_samples/02_hindustani_insp...,/content/finetuned_samples/02_hindustani_inspi...,8.0,4.0,0.184974,0.233880,2147.143051,1357.587027,0.063570,0.015423
3,Veena inspired plucked string improvisation wi...,/content/pretrained_samples/03_veena_inspired_...,/content/finetuned_samples/03_veena_inspired_p...,8.0,4.0,0.185168,0.197594,1008.216970,886.712062,0.040233,0.033505
4,Sitar-like acoustic plucked string phrase with...,/content/pretrained_samples/04_sitar_like_acou...,/content/finetuned_samples/04_sitar_like_acous...,8.0,4.0,0.143697,0.170424,3059.451101,2585.462506,0.134703,0.106688
5,Meditative Indian classical instrumental textu...,/content/pretrained_samples/05_meditative_indi...,/content/finetuned_samples/05_meditative_india...,8.0,4.0,0.211669,0.211487,743.264485,830.028028,0.027896,0.025449


In [30]:
# zip results for download

!zip -r /content/musicgen_results.zip \
  /content/pretrained_samples \
  /content/finetuned_samples \
  /content/comparison.csv \
  /content/musicgen_nsynth_small

from google.colab import files
files.download("/content/musicgen_results.zip")

  adding: content/pretrained_samples/ (stored 0%)
  adding: content/pretrained_samples/01_carnatic_inspired_violin_phrase_with_sustained_acoustic_tone.wav (deflated 4%)
  adding: content/pretrained_samples/04_sitar_like_acoustic_plucked_string_phrase_with_indian_classical_mood.wav (deflated 6%)
  adding: content/pretrained_samples/03_veena_inspired_plucked_string_improvisation_with_raga_like_ornamentati.wav (deflated 4%)
  adding: content/pretrained_samples/02_hindustani_inspired_bansuri_like_melodic_phrase_with_soft_drone.wav (deflated 4%)
  adding: content/pretrained_samples/05_meditative_indian_classical_instrumental_texture_with_sustained_notes.wav (deflated 3%)
  adding: content/pretrained_samples/00_indian_classical_inspired_plucked_string_note_with_tanpura_like_drone_.wav (deflated 4%)
  adding: content/pretrained_samples/manifest.json (deflated 69%)
  adding: content/finetuned_samples/ (stored 0%)
  adding: content/finetuned_samples/01_carnatic_inspired_violin_phrase_with_susta

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>